In [0]:
CREATE TABLE fiap_analytics.gold.fact_geosampa_flooding_events (
  COD_FATO_OCORRENCIA    BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1)
    COMMENT 'Chave substituta numerica da linha',
  COD_DIM_TEMPO          BIGINT    COMMENT 'Data no formato yyyyMMdd',
  COD_DIM_OCORRENCIA     BIGINT    COMMENT 'Chave da dimensao de tipos de ocorrencia',
  COD_DIM_SUBPREFEITURA  BIGINT    COMMENT 'Chave da dimensao de subprefeituras',
  YEAR_PARTITION         STRING    COMMENT 'Ano da ocorrencia',
  MONTH_PARTITION        STRING    COMMENT 'Mes da ocorrencia',
  DAY_PARTITION          STRING    COMMENT 'Dia da ocorrencia',
  NUM_ANO_ORIGEM         INT       COMMENT 'Ano do arquivo de origem (controle da carga incremental)',
  DAT_INGESTAO_BRONZE    TIMESTAMP COMMENT 'Data e hora da ingestao na Bronze',
  DAT_CARGA_FATO         TIMESTAMP COMMENT 'Data e hora da carga desta tabela'
)
COMMENT 'Fato de ocorrencias por tipo e subprefeitura de Sao Paulo (GeoSampa), carga incremental por ano';

In [0]:
CREATE OR REPLACE TEMP VIEW stg_fact_geosampa_flooding_events AS
SELECT
  f.cod_dim_data AS COD_DIM_TEMPO,
  d.COD_DIM_OCORRENCIA,
  a.COD_DIM_SUBPREFEITURA,
  CAST(YEAR(f.dt_ocorrencia) AS STRING)  AS YEAR_PARTITION,
  CAST(MONTH(f.dt_ocorrencia) AS STRING) AS MONTH_PARTITION,
  CAST(DAY(f.dt_ocorrencia) AS STRING)   AS DAY_PARTITION,
  f._source_year                         AS NUM_ANO_ORIGEM,
  f._ingested_at                         AS DAT_INGESTAO_BRONZE
FROM
  fiap_analytics.silver.fact_geosampa_flooding_events f
  LEFT JOIN fiap_analytics.gold.dim_geosampa_type_flooding_events d ON
    f.dc_tipo_ocorrencia = d.DSC_TIPO_OCORRENCIA
  LEFT JOIN fiap_analytics.gold.dim_geosampa_subprefecture_area a ON
    f.sg_subprefeitura = a.SIG_SUBPREFEITURA;

In [0]:
INSERT INTO fiap_analytics.gold.fact_geosampa_flooding_events
  (COD_DIM_TEMPO, COD_DIM_OCORRENCIA, COD_DIM_SUBPREFEITURA,
   YEAR_PARTITION, MONTH_PARTITION, DAY_PARTITION,
   NUM_ANO_ORIGEM, DAT_INGESTAO_BRONZE, DAT_CARGA_FATO)
SELECT
  s.COD_DIM_TEMPO, s.COD_DIM_OCORRENCIA, s.COD_DIM_SUBPREFEITURA,
  s.YEAR_PARTITION, s.MONTH_PARTITION, s.DAY_PARTITION,
  s.NUM_ANO_ORIGEM, s.DAT_INGESTAO_BRONZE, current_timestamp()
FROM stg_fact_geosampa_flooding_events s
  LEFT ANTI JOIN (
    SELECT DISTINCT NUM_ANO_ORIGEM
    FROM fiap_analytics.gold.fact_geosampa_flooding_events
  ) g
    ON s.NUM_ANO_ORIGEM <=> g.NUM_ANO_ORIGEM;

In [0]:
DECLARE OR REPLACE VARIABLE ano_referencia INT;

SET VAR ano_referencia = (SELECT MAX(NUM_ANO_ORIGEM) FROM stg_fact_geosampa_flooding_events);

In [0]:
DECLARE OR REPLACE VARIABLE ano_referencia INT;
SET VAR ano_referencia = (SELECT MAX(NUM_ANO_ORIGEM) FROM stg_fact_geosampa_flooding_events);

DELETE FROM fiap_analytics.gold.fact_geosampa_flooding_events
WHERE NUM_ANO_ORIGEM = ano_referencia;

INSERT INTO fiap_analytics.gold.fact_geosampa_flooding_events
  (COD_DIM_TEMPO, COD_DIM_OCORRENCIA, COD_DIM_SUBPREFEITURA,
   YEAR_PARTITION, MONTH_PARTITION, DAY_PARTITION,
   NUM_ANO_ORIGEM, DAT_INGESTAO_BRONZE, DAT_CARGA_FATO)
SELECT
  COD_DIM_TEMPO, COD_DIM_OCORRENCIA, COD_DIM_SUBPREFEITURA,
  YEAR_PARTITION, MONTH_PARTITION, DAY_PARTITION,
  NUM_ANO_ORIGEM, DAT_INGESTAO_BRONZE, current_timestamp()
FROM stg_fact_geosampa_flooding_events
WHERE NUM_ANO_ORIGEM = ano_referencia;

In [0]:
BEGIN
  IF DAY(from_utc_timestamp(current_timestamp(), 'America/Sao_Paulo')) IN (1, 15) THEN
    OPTIMIZE fiap_analytics.gold.fact_geosampa_flooding_events
    ZORDER BY (COD_DIM_TEMPO, COD_DIM_SUBPREFEITURA);
  END IF;
END